![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, LangChain and Milvus to create and deploy RAG function

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.

## Notebook content

This notebook contains the steps and code to demonstrate support of creating and deploying Retrieval Augmented Generation in watsonx.ai. It introduces commands for data retrieval, knowledge base building & querying, model testing, deploying a RAG solution for general use.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

#### About Retrieval Augmented Generation
Retrieval Augmented Generation (RAG) is a versatile pattern that can unlock a number of use cases requiring factual recall of information, such as querying a knowledge base in natural language.

In its simplest form, RAG requires 3 steps:

- Index knowledge base passages (once)
- Retrieve relevant passage(s) from knowledge base (for every user query)
- Generate a response by feeding retrieved passage into a large language model (for every user query)

## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Test data preparation](#Test-data-preparation)
3. [Set up connectivity information to Milvus](#Set-up-connectivity-information-to-Milvus)
4. [Set up VectorStore with Milvus credentials](#Set-up-VectorStore-with-Milvus-credentials)
5. [Create and deploy RAG solution](#Create-and-deploy-RAG-solution)
6. [Calculate rougeL metric](#Calculate-rougeL-metric)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install rouge-score | tail -n 1
%pip install -U "ibm_watsonx_ai[rag]>=1.4.0" | tail -n 1

In [2]:
import getpass
import os

import wget
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.foundation_models.extensions.rag import VectorStore
from ibm_watsonx_ai.foundation_models.extensions.rag.utils import verbose_search
from ibm_watsonx_ai.foundation_models.prompts import (
    PromptTemplate,
    PromptTemplateManager,
)
from ibm_watsonx_ai.helpers import DataConnection
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer

### Define the watsonx.ai credentials
Use the code cell below to define the watsonx.ai credentials that are required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">Managing user API keys</a>.

In [3]:
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your api key and hit enter: "),
)

### Defining the project ID

The Foundation Model requires project ID that provides the context for the call. We will obtain the ID from the project in which this notebook runs. Otherwise, please provide the project ID.

In [4]:
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id and hit enter: ")

### Defining the space ID
Deployed functions are available on deployment spaces. RAG we will create, will be a deployed function. You need to provide space id.

In [5]:
space_id = input("Please enter your space_id and hit enter: ")

### Initialize client
Create an instance of `APIClient` and set the default project.

In [6]:
client = APIClient(credentials, project_id=project_id)

<a id="Test-data-preparation"></a>
## Test data preparation

### Defining the prompt ID

We will use PromptTemplate to create a template for our RAG LLM query. If you don't have the PromptTemplate created in your project, this code will create an example one.

In [7]:
prompt_id = (
    input(
        "Please enter your prompt template asset id and hit enter, if not provided, a new one would be created: "
    )
    or None
)

if prompt_id is None:
    PROMPT_INSTRUCTION = """
    Use the following pieces of documents to answer the question
    at the end. If you don't know the answer, just say that you
    don't know, don't try to make up an answer. Use three sentences
    maximum. Keep the answer as concise as possible. do not include
    question in your response.Your answers should not include any
    harmful, unethical, racist, sexist, toxic, dangerous, or illegal
    content. Please ensure that your responses are socially unbiased
    and positive in nature.\nPlease provide a concise professional
    response.
    """
    prompt_mgr = PromptTemplateManager(credentials=credentials, project_id=project_id)
    prompt_template = PromptTemplate(
        name="RAG_prompt_template",
        model_id=client.foundation_models.TextModels.LLAMA_3_3_70B_INSTRUCT,
        input_variables=["question", "reference_documents"],
        instruction=PROMPT_INSTRUCTION,
        input_text="{reference_documents}\nQuestion:{question}\nAnswer:",
    )
    stored_prompt_template = prompt_mgr.store_prompt(prompt_template=prompt_template)
    prompt_id = stored_prompt_template.prompt_id

### Build up knowledge base

The current state-of-the-art in RAG is to create dense vector representations of the knowledge base in order to calculate the semantic similarity to a given user query.

We can generate dense vector representations using embedding models. In this notebook, we use IBM's <a href="https://www.ibm.com/products/watsonx-ai/foundation-models#Embedding+model+library">IBM_SLATE_30M_ENG</a> model to embed both the knowledge base passages and user queries.

A vector database is optimized for dense vector indexing and retrieval. This notebook uses <a href="https://python.langchain.com/docs/integrations/vectorstores/Milvus#basic-example" target="_blank" rel="noopener no referrer">Milvus</a>, an open-source vector database.

The dataset we are using is already split into self-contained passages that can be ingested by Milvus. 

The size of each passage is limited by the embedding model's context window (which is 512 tokens for `IBM Slate 30M`).

### Load knowledge base documents

Load set of documents used further to build knowledge base and store them as a project asset.

In [8]:
filename = "psgs.tsv"
url = f"https://raw.github.com/IBM/watsonx-ai-samples/master/cloud/data/RAG/{filename}"
if not os.path.isfile(filename):
    wget.download(url)

asset_details = client.data_assets.create(name=filename, file_path=filename)

Creating data asset...
SUCCESS


### Read and prepare documents
Read documents using `DataConnection` and prepare them for vector database ingestion by combining title and text.

In [9]:
data_connection = DataConnection(data_asset_id=client.data_assets.get_id(asset_details))
data_connection.set_client(client)
documents = data_connection.read(csv_separator="\t")

  Using cached pyarrow-24.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-macosx_12_0_arm64.whl (35.0 MB)


In [10]:
documents["indextext"] = documents["title"].astype(str) + "\n" + documents["text"]
documents = documents[:1000]
documents.head()

,id,text,title,indextext
0,1,History of Idaho - wikipedia History of Idaho ...,History of Idaho,History of Idaho\nHistory of Idaho - wikipedia...
1,2,"1957 . Location Cataldo , Idaho Built 1848 Arc...",History of Idaho,"History of Idaho\n1957 . Location Cataldo , Id..."
2,3,"of the Columbia was created in June 1816 , and...",History of Idaho,History of Idaho\nof the Columbia was created ...
3,4,"Canyon , he concluded that water transport was...",History of Idaho,"History of Idaho\nCanyon , he concluded that w..."
4,5,"1842 , Father Pierre - Jean De Smet , with Fr....",History of Idaho,"History of Idaho\n1842 , Father Pierre - Jean ..."


### Create an embedding function for VectorStore

Note that you can feed a custom embedding function to be used by Milvus. The performance of Milvus may differ depending on the embedding model used. 

In [11]:
embeddings = Embeddings(
    model_id=client.foundation_models.EmbeddingModels.SLATE_30M_ENGLISH_RTRVR_V2,
    credentials=credentials,
    project_id=project_id,
)

<a id="Set-up-connectivity-information-to-Milvus"></a>
## Set up connectivity information to Milvus

**This notebook focuses on self-managed Milvus cluster using <a href="https://cloud.ibm.com/docs/watsonxdata?topic=watsonxdata-adding-milvus-service" target="_blank" rel="noopener no referrer">IBM watsonx.data.</a>**

The following cell retrieves the Milvus username, password, host and port from the environment if available and prompts you otherwise.

You can provide a connection asset ID to read all required connection data from it. Before doing so, make sure that connection asset was created in your project.

In [12]:
connection_id = (
    input(
        "Provide connection asset ID in your project. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if connection_id is None:
    try:
        username = os.environ["USERNAME"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")

    try:
        password = os.environ["PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")

    try:
        host = os.environ["HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")

    try:
        port = os.environ["PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")

    try:
        ssl = os.environ["SSL"]
    except KeyError:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    connection_id = client.connections.get_id(details)

<a id="Set-up-VectorStore-with-Milvus-credentials"></a>
## Set up VectorStore with Milvus credentials

Create a VectorStore class that automatically detects the database type (in our case it will be Milvus) and allows us to add, search and delete documents.

It works as a wrapper for LangChain VectorStore classes. You can customize the settings as long as it is supported. Consult the LangChain documentation for more information about <a href="https://api.python.langchain.com/en/latest/vectorstores/langchain_community.vectorstores.milvus.Milvus.html" target="_blank" rel="noopener no referrer">Milvus</a> connector.

Provide the name of your Milvus index for subsequent operations:

In [13]:
index_name = input("Please enter Milvus index name and hit enter: ")

In [14]:
vector_store = VectorStore(
    client=client,
    embeddings=embeddings,
    connection_id=connection_id,
    index_name=index_name,
    secure=True,
)

In [15]:
client.set.default_space(space_id)

Unsetting the project_id ...


'SUCCESS'

### Embed and index documents with Milvus

**Note: Could take several minutes if you don't have pre-built indices**

In [16]:
texts = documents.indextext.tolist()
metadatas = [
    {"title": title, "id": doc_id}
    for (title, doc_id) in zip(documents.title, documents.id)
]
docs_to_add = [
    Document(page_content=text, metadata=metadata)
    for text, metadata in zip(texts, metadatas)
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
docs_to_add_split = text_splitter.split_documents(docs_to_add)

ids = vector_store.add_documents(docs_to_add_split, batch_size=200)

Verify the number of documents loaded into the Milvus.

In [17]:
doc_count = vector_store.count()
doc_count

4051

Let's search for an example document as a sample. Note the embedding in the vector field, that was generated with the sentence transformer.

In [18]:
vector_store.search("United States of America", k=5, verbose=True)

**Question:** United States of America

,page_content,pk,id,title
0,", D.C. States / Territories Alabama Alaska Ame...",71682bac720e55707b03cb834a9e1cc7b3768fcbe44fb0...,639,"United States Senate elections, 2018"
1,"United States , 1797 -- 1801 1st Vice Presiden...",65fe7b6f1a088e880b2ef978db5cfa826b1aa5b13d7762...,918,Founding Fathers of the United States
2,"the United States of America , which was recog...",40ce604ff3d03e5b38c52bcef4fc75a9df77b4470f9b9c...,521,British colonization of the Americas
3,1937 1938 1939 1940 1941 1942 1943 1944 1945 1...,bb2c2fb8b6617c060bccf8cedfc76c8661532923ac0afa...,22,History of Idaho
4,of Congress further identifies the Articles of...,b77133e015fe4352a25ff4efc85b0cb1edc2321298e8f4...,893,Founding Fathers of the United States


[Document(metadata={'title': 'United States Senate elections, 2018', 'id': 639, 'pk': '71682bac720e55707b03cb834a9e1cc7b3768fcbe44fb0a7a2b0b5ee48365662'}, page_content=', D.C. States / Territories Alabama Alaska American Samoa Arizona Arkansas California Colorado Connecticut Delaware Florida Georgia Guam Hawaii Idaho Illinois Indiana Iowa Kansas Kentucky Louisiana Maine Maryland Massachusetts Michigan Minnesota Mississippi Missouri Montana Nebraska Nevada New Hampshire New Jersey New Mexico New York North Carolina North Dakota Ohio Oklahoma Oregon Pennsylvania Puerto Rico'),
 Document(metadata={'title': 'Founding Fathers of the United States', 'id': 918, 'pk': '65fe7b6f1a088e880b2ef978db5cfa826b1aa5b13d776264b29a678b5d147504'}, page_content='United States , 1797 -- 1801 1st Vice President of the United States , 1789 -- 1797 U.S. Ambassador to the United Kingdom , 1785 -- 1788 U.S. Ambassador to the Netherlands , 1782 -- 1788 Delegate , Second Continental Congress , 1775 -- 1778 Delegat

<a id="Create-and-deploy-RAG-solution"></a>
## Create and deploy RAG solution

### Define ai service code

Deployed function for RAG should implement the functionality of retrieval and augmenting the prompt for the LLM model.
Function defined can be used as an example. To modify the deployed function behaviour, change the values in `custom`.

In [19]:
custom = {
    "url": credentials.url,
    "space_id": space_id,
    "retriever": {"method": "simple", "number_of_chunks": 5},
    "vector_store": vector_store.to_dict(),
    "prompt_template_text": "\n    Use the following pieces of documents to answer the question\n    at the end. If you don't know the answer, just say that you\n    don't know, don't try to make up an answer. Use three sentences\n    maximum. Keep the answer as concise as possible. do not include\n    question in your response.Your answers should not include any\n    harmful, unethical, racist, sexist, toxic, dangerous, or illegal\n    content. Please ensure that your responses are socially unbiased\n    and positive in nature.\nPlease provide a concise professional\n    response.\n    \n\n{reference_documents}\nQuestion:{question}\nAnswer:",
    "context_template_text": None,
    "model": {
        "model_id": "ibm/granite-4-h-small",
        "params": {
            "decoding_method": "greedy",
            "min_new_tokens": 1,
            "max_new_tokens": 200,
        },
        "project_id": None,
        "space_id": space_id,
    },
    "inference_function_params": {},
}


def deployable_ai_service(context, **custom):
    """
    Deployed function.

    Input schema:
    payload = {
        'values': ['question 1', 'question 2']
    }

    Output schema:
    result = {
        'predictions': [
            {
                'fields': ['answer', 'reference_documents'],
                'values': [
                    ['answer 1', [ {'page_content': 'page content 1',
                                    'metadata':     'metadata 1'} ]],
                    ['answer 2', [ {'page_content': 'page content 2',
                                    'metadata':     'metadata 2'} ]]
                ]
            }
        ]
    }
    """

    from ibm_watsonx_ai import APIClient, Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference
    from ibm_watsonx_ai.foundation_models.extensions.rag import Retriever, VectorStore
    from ibm_watsonx_ai.foundation_models.extensions.rag.pattern.prompt_builder import (
        build_prompt,
    )
    from ibm_watsonx_ai.metanames import GenTextParamsMetaNames

    client = APIClient(
        credentials=Credentials(url=custom.get("url"), token=context.generate_token()),
        space_id=custom.get("space_id"),
    )
    vector_store = VectorStore.from_dict(client=client, data=custom["vector_store"])
    retriever = Retriever.from_vector_store(
        vector_store=vector_store, init_parameters=custom["retriever"]
    )
    prompt_template_text = custom["prompt_template_text"]
    context_template_text = custom["context_template_text"]
    model = ModelInference(api_client=client, **custom["model"])
    model_specs = client.foundation_models.get_model_specs(model_id=model.model_id)
    model_max_new_tokens = (model.params or {}).get(
        GenTextParamsMetaNames.MAX_NEW_TOKENS, 20
    )
    model_max_input_tokens = (
        model_specs["model_limits"]["max_sequence_length"] - model_max_new_tokens
    )

    def generate(context):
        client.set_token(context.get_token())
        payload = context.get_json()
        result = {"predictions": [{"fields": ["answer", "reference_documents"]}]}

        all_prompts = []
        all_retrieved_docs = []

        for question in payload["values"]:
            retrieved_docs = retriever.retrieve(query=question)
            all_retrieved_docs.append(retrieved_docs)
            reference_documents = [doc.page_content for doc in retrieved_docs]

            prompt_input_text = build_prompt(
                prompt_template_text=prompt_template_text,
                context_template_text=context_template_text,
                question=question,
                reference_documents=reference_documents,
                model_max_input_tokens=model_max_input_tokens,
            )
            all_prompts.append(prompt_input_text)

        answers = [model.generate_text(prompt=prompt) for prompt in all_prompts]

        predictions = [
            [
                answer,
                [
                    {"page_content": doc.page_content, "metadata": doc.metadata}
                    for doc in retrieved_docs
                ],
            ]
            for answer, retrieved_docs in zip(answers, all_retrieved_docs)
        ]

        result["predictions"][0]["values"] = predictions

        return {"body": result}

    return generate

### Test the function locally

To test our solution we can query the function locally without deploying.

In [20]:
questions_and_answers = {
    "what are the names of founding fathers of the united states?": "Thomas Jefferson::James Madison::John Jay::George Washington::John Adams::Benjamin Franklin::Alexander Hamilton",
    "who played in the super bowl in 2013?": "Baltimore Ravens::San Francisco 49ers",
    "when did bucharest become the capital of romania?": "1862",
}

Define a helper function for formatting the response:

In [21]:
def print_rag_response(response):
    for question, (answer, reference_docs) in zip(
        questions_and_answers.keys(), response["predictions"][0]["values"]
    ):
        verbose_search(question, [Document(**d) for d in reference_docs])
        display(Markdown(f"**Answer:** {answer}"))

Questions have to be provided in the payload that have format provided below.

In [22]:
payload = {"values": list(questions_and_answers.keys())}

In [23]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)
generate_function = deployable_ai_service(context=context, **custom)

Unsetting the space_id ...
Unsetting the project_id ...


In [24]:
context.request_payload_json = payload
response = generate_function(context=context)
print_rag_response(response["body"])

**Question:** what are the names of founding fathers of the united states?

,page_content,pk,id,title
0,Founding Fathers of the United States,360e0cb536b2121387f686b2d1092edb88fa1d9232b122...,878,Founding Fathers of the United States
1,Founding Fathers of the United States,3d0db85f927e5d125275e69efab525276cbfa44159be1d...,879,Founding Fathers of the United States
2,Founding Fathers of the United States,e2c4763d51b61b39f586d110d9105a5b89cc1f49c908d5...,880,Founding Fathers of the United States
3,Founding Fathers of the United States,310caa35def2f771b35631f7e5db9280f3b8c661ac67e1...,881,Founding Fathers of the United States
4,Founding Fathers of the United States,909594651176c93a1469f71d72b1a28199c636f9b8392f...,882,Founding Fathers of the United States


**Answer:**  The Founding Fathers of the United States include George Washington, John Adams, Thomas Jefferson, James Madison, Alexander Hamilton, Benjamin Franklin, and John Jay.

**Question:** who played in the super bowl in 2013?

,page_content,pk,id,title
0,"responded to the claim on Twitter in jest , tw...",a1d0e756f81f7e0059aa1c113d886a39b07d6053c0662a...,848,Super Bowl XLVII
1,Super Bowl XLVII - wikipedia Super Bowl XLVII ...,c217b371bbbc044630f866cb154e75937dbdb9240c92da...,818,Super Bowl XLVII
2,Broadcast Schedule : NFL Super Bowl XLVII -- 2...,0b5e6cc429a36c859d7fb0b5ea39553b09cffa239a496b...,863,Super Bowl XLVII
3,: Super Bowl 2012 National Football League sea...,df487f7df65ec4873760902ec0d04aee8bb30f04565e59...,876,Super Bowl XLVII
4,Opponents Announced '' . NewOrleansSaints.com ...,60b0a899bbf8b2633e33e8df73d552b1ed6c3f3e9e3815...,856,Super Bowl XLVII


**Answer:**  The Baltimore Ravens and the San Francisco 49ers played in Super Bowl XLVII in 2013.

**Question:** when did bucharest become the capital of romania?

,page_content,pk,id,title
0,destroying a third of the city . Ottoman massa...,6d6b2837623cf2f230a2b682cdbb76bdef6363499c8db6...,948,Bucharest
1,documents in 1459 . It became the capital of R...,13bd23f66a114ffcb372f4235705f660d6c2d99ac8cebd...,944,Bucharest
2,exist . Bucharest 's population experienced tw...,e711e2a5de7104dc86bbe7eb23918df2396f8961ed92a1...,965,Bucharest
3,. I.C. Brătianu Boulevard in the 1930s Between...,d019524154381a11b60d6b8fe9c65b71b872b1ec918994...,948,Bucharest
4,Bucharest,e4a9ac850849199f157559dccca93fa51c08171efb9676...,942,Bucharest


**Answer:**  Bucharest became the capital of Romania in 1862.

### Deploy RAGPattern

Deployment is done by storing the defined RAG function and then by creating a deployed asset. It would be now accessed as an endpoint that we can run.

In [25]:
sw_spec_id = client.software_specifications.get_id_by_name("genai-A25-py3.12")

meta_props = {
    client.repository.AIServiceMetaNames.NAME: "RAG AI service SDK",
    client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}

stored_ai_service_details = client.repository.store_ai_service(
    deployable_ai_service, meta_props
)

In [26]:
ai_service_id = client.repository.get_ai_service_id(stored_ai_service_details)
ai_service_id

'019df7aa-01c0-77ea-9da6-8aafd5489bea'

In [27]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "RAG pattern AI service",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
    client.deployments.ConfigurationMetaNames.CUSTOM: custom,
}

deployment_details = client.deployments.create(ai_service_id, meta_props)



######################################################################################

Synchronous deployment creation for id: '019df7aa-01c0-77ea-9da6-8aafd5489bea' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
......
Note: AI service function with **kwargs as parameter is deprecated and will be discontinued in future release. Use deployment parameters instead.

ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='019df7ab-78f3-70f7-be4c-e103aa3514ea'
-----------------------------------------------------------------------------------------------




### Test the deployed function

RAG service is now deployed on our space. To test our solution we can run the cell below. Questions have to be provided in the payload that have format provided below.

In [28]:
deployment_id = client.deployments.get_id(deployment_details)

In [29]:
response = client.deployments.run_ai_service(deployment_id, payload)
print_rag_response(response)

**Question:** what are the names of founding fathers of the united states?

,page_content,pk,id,title
0,Founding Fathers of the United States,360e0cb536b2121387f686b2d1092edb88fa1d9232b122...,878,Founding Fathers of the United States
1,Founding Fathers of the United States,3d0db85f927e5d125275e69efab525276cbfa44159be1d...,879,Founding Fathers of the United States
2,Founding Fathers of the United States,e2c4763d51b61b39f586d110d9105a5b89cc1f49c908d5...,880,Founding Fathers of the United States
3,Founding Fathers of the United States,310caa35def2f771b35631f7e5db9280f3b8c661ac67e1...,881,Founding Fathers of the United States
4,Founding Fathers of the United States,909594651176c93a1469f71d72b1a28199c636f9b8392f...,882,Founding Fathers of the United States


**Answer:**  The founding fathers of the United States include prominent figures such as George Washington, Thomas Jefferson, Benjamin Franklin, John Adams, James Madison, and Alexander Hamilton.

**Question:** who played in the super bowl in 2013?

,page_content,pk,id,title
0,"responded to the claim on Twitter in jest , tw...",a1d0e756f81f7e0059aa1c113d886a39b07d6053c0662a...,848,Super Bowl XLVII
1,Super Bowl XLVII - wikipedia Super Bowl XLVII ...,c217b371bbbc044630f866cb154e75937dbdb9240c92da...,818,Super Bowl XLVII
2,Broadcast Schedule : NFL Super Bowl XLVII -- 2...,0b5e6cc429a36c859d7fb0b5ea39553b09cffa239a496b...,863,Super Bowl XLVII
3,: Super Bowl 2012 National Football League sea...,df487f7df65ec4873760902ec0d04aee8bb30f04565e59...,876,Super Bowl XLVII
4,Opponents Announced '' . NewOrleansSaints.com ...,60b0a899bbf8b2633e33e8df73d552b1ed6c3f3e9e3815...,856,Super Bowl XLVII


**Answer:**  The Baltimore Ravens and San Francisco 49ers played in the 2013 Super Bowl.

**Question:** when did bucharest become the capital of romania?

,page_content,pk,id,title
0,destroying a third of the city . Ottoman massa...,6d6b2837623cf2f230a2b682cdbb76bdef6363499c8db6...,948,Bucharest
1,documents in 1459 . It became the capital of R...,13bd23f66a114ffcb372f4235705f660d6c2d99ac8cebd...,944,Bucharest
2,exist . Bucharest 's population experienced tw...,e711e2a5de7104dc86bbe7eb23918df2396f8961ed92a1...,965,Bucharest
3,. I.C. Brătianu Boulevard in the 1930s Between...,d019524154381a11b60d6b8fe9c65b71b872b1ec918994...,948,Bucharest
4,Bucharest,e4a9ac850849199f157559dccca93fa51c08171efb9676...,942,Bucharest


**Answer:**  Bucharest became the capital of Romania in 1862.

<a id="Calculate-rougeL-metric"></a>
## Calculate rougeL metric
Calculate rougeL recall score to verify expected answer presence in generated response.

In [30]:
text_responses = [v[0] for v in response["predictions"][0]["values"]]
targets = [answer for answer in questions_and_answers.values()]

In [31]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
scores = [
    scorer.score(target, prediction)
    for target, prediction in zip(targets, text_responses)
]
mean_rougeL = sum([s["rougeL"].recall for s in scores]) / len(questions_and_answers)

print(f"Mean rougeL recall score: {mean_rougeL}")

Mean rougeL recall score: 0.8095238095238094


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Dominik Zimny (Former)**, Software Engineer at IBM watsonx.ai

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai.

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.